# Train Decision Segmenter (OPF)

Fine-tune the OpenAI Privacy Filter (1.5B params) as a 22-class judicial decision segmenter.

**Requirements:** GPU runtime (T4 or better, 15GB+ VRAM)

Go to **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Switch to GPU runtime first!"

In [ ]:
!pip install -q "opf @ git+https://github.com/openai/privacy-filter.git"

In [ ]:
# Download OPF base checkpoint (~2.8GB)
from opf._common.checkpoint_download import ensure_default_checkpoint
ensure_default_checkpoint()

In [ ]:
# Clone repo and get prepared data
!git clone --depth 1 --branch claude/epic-clarke-9uaaQ https://github.com/franklinbaldo/causaganha.git /content/causaganha

In [ ]:
import os
os.chdir("/content/causaganha")
os.environ["PYTHONPATH"] = "/content/causaganha"

!pip install -q ibis-framework[duckdb] structlog pyarrow

In [ ]:
# Prepare JSONL data from labeled parquet
!python scripts/train_decision_segmenter.py \
    --labeled-parquet data/benchmark/segmenter_training.parquet \
    --output-dir /content/models/decision_segmenter \
    --prepare-only

In [ ]:
# Train with OPF on T4 (batch_size=1 + n-ctx=512 to fit in 15GB VRAM)
# On A100 (40GB): increase to --batch-size 4 --n-ctx 1024
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python -m opf train /content/models/decision_segmenter/train.jsonl \
    --validation-dataset /content/models/decision_segmenter/val.jsonl \
    --label-space-json /content/models/decision_segmenter/label_space.json \
    --output-dir /content/models/decision_segmenter/best \
    --device cuda \
    --epochs 3 \
    --batch-size 1 \
    --n-ctx 512

In [ ]:
# Evaluate on test set (eval reads label space from the checkpoint)
!python -m opf eval /content/models/decision_segmenter/test.jsonl \
    --checkpoint /content/models/decision_segmenter/best \
    --device cuda \
    --per-class \
    --metrics-out /content/models/decision_segmenter/test_metrics.json

In [ ]:
import json

metrics = json.load(open("/content/models/decision_segmenter/test_metrics.json"))

print("=" * 60)
print("TEST METRICS")
print("=" * 60)
macro = metrics.get("macro avg", {})
print(f"Macro F1: {macro.get('f1-score', 0):.3f}")
disp = metrics.get("sec_dispositivo", {})
print(f"sec_dispositivo F1: {disp.get('f1-score', 0):.3f}")
print()
for k, v in metrics.items():
    if isinstance(v, dict) and "f1-score" in v and k not in ("macro avg", "weighted avg", "micro avg"):
        print(f"  {k:<22} P={v.get('precision',0):.2f}  R={v.get('recall',0):.2f}  F1={v.get('f1-score',0):.2f}  n={v.get('support',0)}")

In [ ]:
# Download trained model (save to Google Drive or download directly)
!tar -czf /content/decision_segmenter.tar.gz -C /content/models/decision_segmenter/best .
print(f"Model saved: /content/decision_segmenter.tar.gz")

# Optional: mount Drive and copy
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/decision_segmenter.tar.gz /content/drive/MyDrive/
# !cp /content/models/decision_segmenter/test_metrics.json /content/drive/MyDrive/